# Embedded AI Final Project

#### 20223265 조해영, 20214904 조성

0. 환경 설정

In [1]:
import json
import platform
import sys

import torch
import torchvision

# ROCm(AMD Radeon) 빌드의 PyTorch도 torch.cuda API를 그대로 쓰므로 device 문자열은 "cuda"이다.
if torch.version.hip:
    GPU_BACKEND = f"ROCm/HIP {torch.version.hip} (AMD)"
elif torch.version.cuda:
    GPU_BACKEND = f"CUDA {torch.version.cuda} (NVIDIA)"
else:
    GPU_BACKEND = "none (CPU-only PyTorch build)"

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("GPU backend:", GPU_BACKEND)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_name = gpu_props.name
    gpu_arch = getattr(gpu_props, "gcnArchName", None) or f"sm_{gpu_props.major}{gpu_props.minor}"
    print(f"GPU: {gpu_name} ({gpu_arch}, {gpu_props.total_memory / 1024**3:.1f} GiB)")
else:
    DEVICE = "cpu"
    gpu_name = gpu_arch = None
    print("[warn] No GPU detected; running on CPU.")
    if not torch.version.hip and not torch.version.cuda:
        print(
            "       This PyTorch build is CPU-only. For an AMD Radeon RX 6800 (gfx1030) on Windows:\n"
            '       pip install --index-url https://stable.repo.amd.com/rocm/whl-next/ '
            '"torch[device-gfx1030]" "torchvision[device-gfx1030]"'
        )

ENV_INFO = {
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "gpu_backend": GPU_BACKEND,
    "device": DEVICE,
    "gpu_name": gpu_name,
    "gpu_arch": gpu_arch,
    "platform": platform.platform(),
}
print()
print(json.dumps(ENV_INFO, indent=2, ensure_ascii=False))

python: 3.13.9
torch: 2.14.0+cpu
torchvision: 0.29.0+cpu
GPU backend: none (CPU-only PyTorch build)
GPU available: False
[warn] No GPU detected; running on CPU.
       This PyTorch build is CPU-only. For an AMD Radeon RX 6800 (gfx1030) on Windows:
       pip install --index-url https://stable.repo.amd.com/rocm/whl-next/ "torch[device-gfx1030]" "torchvision[device-gfx1030]"

{
  "python_version": "3.13.9",
  "torch_version": "2.14.0+cpu",
  "torchvision_version": "0.29.0+cpu",
  "gpu_backend": "none (CPU-only PyTorch build)",
  "device": "cpu",
  "gpu_name": null,
  "gpu_arch": null,
  "platform": "Windows-11-10.0.26200-SP0"
}


### 1. 모델 호출

Classification을 위한 ViT를 호출한다

In [2]:
import timm
from timm.data import create_transform, resolve_model_data_config

MODEL_NAME = "vit_base_patch16_224"
NUM_CLASSES = 1000


def load_model(model_name=MODEL_NAME, device="cpu"):
    model = timm.create_model(model_name, num_classes=NUM_CLASSES, pretrained=True)
    model.to(device)
    model.eval()
    return model


def build_eval_transform(model):
    data_config = resolve_model_data_config(model)
    return create_transform(**data_config, is_training=False), data_config


model = load_model(device=DEVICE)
eval_transform, data_config = build_eval_transform(model)

print("model:", MODEL_NAME)
print("device:", DEVICE)
print("preprocessing:", data_config)

model: vit_base_patch16_224
device: cpu
preprocessing: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}


In [3]:
total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
state_dict_size_bytes = sum(
    tensor.numel() * tensor.element_size() for tensor in model.state_dict().values()
)
state_dict_size_mib = state_dict_size_bytes / (1024 ** 2)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"State dict tensor size: {state_dict_size_mib:.2f} MiB ({state_dict_size_bytes:,} bytes)")

Total parameters: 86,567,656
Trainable parameters: 86,567,656
State dict tensor size: 330.23 MiB (346,270,624 bytes)


### ImageNet calibration / test 데이터

In [4]:
import re
import shutil
import tarfile
import tempfile
from pathlib import Path

IMAGENET_ARCHIVE = Path("./val_images.tar.gz")
IMAGENET_ROOT = Path("./data/imagenet1k/val")
NUM_CLASSES = 1000


def ensure_imagenet_prepared(archive_path, dataset_root):
    if dataset_root.is_dir():
        print(f"Using existing ImageNet files: {dataset_root.resolve()}")
        return
    if not archive_path.is_file():
        raise FileNotFoundError(f"ImageNet archive was not found: {archive_path.resolve()}")

    dataset_root.parent.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path.resolve()}...")
    with tempfile.TemporaryDirectory(dir=dataset_root.parent) as temp_dir:
        extract_dir = Path(temp_dir) / "payload"
        extract_dir.mkdir()
        with tarfile.open(archive_path, "r:*") as archive:
            def members_with_progress():
                for count, member in enumerate(archive, start=1):
                    if count % 1000 == 0:
                        print(f"Extracting archive entries: {count:,}", flush=True)
                    yield member

            archive.extractall(extract_dir, members=members_with_progress(), filter="data")

        candidates = [
            directory for directory in [extract_dir, *extract_dir.rglob("*")]
            if directory.is_dir()
            and sum(1 for child in directory.iterdir() if child.is_dir()) == NUM_CLASSES
        ]
        if len(candidates) == 1:
            shutil.move(str(candidates[0]), str(dataset_root))
        elif not candidates:
            images = [path for path in extract_dir.rglob("*") if path.suffix.lower() in {".jpg", ".jpeg", ".png"} and path.is_file()]
            labeled_images = [(path, re.search(r"_(n\d{8})$", path.stem)) for path in images]
            if not labeled_images or any(match is None for _, match in labeled_images):
                raise ValueError("Expected class folders or image filenames ending in _nXXXXXXXX.")
            if len({match.group(1) for _, match in labeled_images}) != NUM_CLASSES:
                raise ValueError("Expected images from 1,000 ImageNet classes.")
            organized_dir = Path(temp_dir) / "organized"
            for count, (path, match) in enumerate(labeled_images, start=1):
                class_dir = organized_dir / match.group(1)
                class_dir.mkdir(parents=True, exist_ok=True)
                destination = class_dir / path.name
                if destination.exists():
                    raise ValueError(f"Duplicate image filename: {path.name}")
                shutil.move(str(path), str(destination))
                if count % 1000 == 0:
                    print(f"Organizing images: {count:,}/{len(labeled_images):,}", flush=True)
            shutil.move(str(organized_dir), str(dataset_root))
        else:
            raise ValueError("Found multiple directories containing 1,000 class folders.")

    print(f"ImageNet validation files: {dataset_root.resolve()}")


ensure_imagenet_prepared(IMAGENET_ARCHIVE, IMAGENET_ROOT)

Using existing ImageNet files: C:\Users\82105\3-2\임베디드 AI\Embedded-AI-Final-Project\data\imagenet1k\val


In [7]:
import re
from pathlib import Path

from timm.data.imagenet_info import ImageNetInfo
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

IMAGENET_ROOT = Path("./data/imagenet1k/val")
CALIBRATION_RATIO = 0.2
BATCH_SIZE = 32
NUM_WORKERS = 0  # Reliable in notebooks on Windows.
SEED = 42
NUM_CLASSES = 1000

if not IMAGENET_ROOT.is_dir():
    raise FileNotFoundError(f"ImageNet folder was not found: {IMAGENET_ROOT.resolve()}")

imagenet_info = ImageNetInfo()
synsets = imagenet_info.label_names()
class_names = imagenet_info.label_descriptions()
class_folder_names = []
used_names = set()
for description in class_names:
    base_name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", description).strip(" .")
    if not base_name or re.fullmatch(r"(?i:CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9])(?:\..*)?", base_name):
        base_name = f"class {base_name}"
    folder_name = base_name
    suffix = 2
    while folder_name.casefold() in used_names:
        folder_name = f"{base_name} ({suffix})"
        suffix += 1
    class_folder_names.append(folder_name)
    used_names.add(folder_name.casefold())


def rename_imagenet_class_folders(dataset_root):
    folders = {entry.name: entry for entry in dataset_root.iterdir() if entry.is_dir()}
    if len(folders) != NUM_CLASSES:
        raise ValueError(f"Expected 1,000 class folders, found {len(folders)} under {dataset_root}.")

    renames = []
    matched = set()
    for class_id, (synset, folder_name) in enumerate(zip(synsets, class_folder_names)):
        candidates = {synset, str(class_id), folder_name} & folders.keys()
        if len(candidates) != 1:
            raise ValueError(f"Expected exactly one folder for class {class_id} ({folder_name}), found {sorted(candidates)}.")
        current_name = next(iter(candidates))
        if current_name in matched:
            raise ValueError(f"Folder {current_name} matches more than one class.")
        matched.add(current_name)
        if current_name != folder_name:
            renames.append((folders[current_name], dataset_root / folder_name))

    if len(matched) != len(folders):
        raise ValueError("Some folders do not match an ImageNet class.")
    for source, destination in renames:
        if destination.exists():
            raise FileExistsError(f"Cannot rename {source}: destination already exists: {destination}")
    for source, destination in renames:
        source.rename(destination)
    print(f"Class folders renamed: {len(renames)}")


class ImageNetClassImageFolder(ImageFolder):
    def find_classes(self, directory):
        classes = class_folder_names
        return classes, {class_name: class_id for class_id, class_name in enumerate(classes)}


def split_calibration_test(dataset, calibration_ratio=CALIBRATION_RATIO, seed=SEED):
    if not 0 < calibration_ratio < 1:
        raise ValueError("Calibration ratio must be between 0 and 1.")
    rng = torch.Generator().manual_seed(seed)
    class_indices = {class_id: [] for class_id in range(NUM_CLASSES)}
    for index, class_id in enumerate(dataset.targets):
        class_indices[class_id].append(index)

    calibration_indices = []
    val_indices = []
    calibration_counts = []
    val_counts = []
    for class_id, indices in class_indices.items():
        if len(indices) < 2:
            raise ValueError(f"Class {class_id} needs at least 2 images to split into both datasets.")
        shuffled = torch.randperm(len(indices), generator=rng).tolist()
        split_at = round(len(indices) * calibration_ratio)
        split_at = min(max(split_at, 1), len(indices) - 1)
        calibration_indices.extend(indices[position] for position in shuffled[:split_at])
        val_indices.extend(indices[position] for position in shuffled[split_at:])
        calibration_counts.append(split_at)
        val_counts.append(len(indices) - split_at)

    return Subset(dataset, calibration_indices), Subset(dataset, val_indices), calibration_counts, val_counts


rename_imagenet_class_folders(IMAGENET_ROOT)
dataset = ImageNetClassImageFolder(IMAGENET_ROOT, transform=eval_transform)
calibration_dataset, val_dataset, calibration_counts, val_counts = split_calibration_test(dataset)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE == "cuda",
}
calibration_loader = DataLoader(calibration_dataset, shuffle=False, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)

print(f"Dataset: {IMAGENET_ROOT.resolve()}")
print(f"ImageNet classes: {len(dataset.classes)}")
print(f"Calibration images: {len(calibration_dataset)}")
print(f"Test images: {len(val_dataset)}")
print(f"Per-class calibration counts: {sorted(set(calibration_counts))}")
print(f"Per-class test counts: {sorted(set(val_counts))}")
print(f"Image transform: {eval_transform}")

Class folders renamed: 1000
Dataset: C:\Users\82105\3-2\임베디드 AI\Embedded-AI-Final-Project\data\imagenet1k\val
ImageNet classes: 1000
Calibration images: 10000
Test images: 40000
Per-class calibration counts: [10]
Per-class test counts: [40]
Image transform: Compose(
    Resize(size=248, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.5000, 0.5000, 0.5000]), std=tensor([0.5000, 0.5000, 0.5000]))
)


### 3. Baseline 모델 실행

In [ ]:
if dataset.classes != class_folder_names:
    raise ValueError("ImageNet class folders do not match the model's class order.")

model.eval()
correct = 0
total = 0
class_correct = torch.zeros(NUM_CLASSES, dtype=torch.int64)
class_total = torch.zeros(NUM_CLASSES, dtype=torch.int64)

with torch.inference_mode():
    for images, labels in val_loader:
        images = images.to(DEVICE, non_blocking=DEVICE == "cuda")
        labels = labels.to(DEVICE, non_blocking=DEVICE == "cuda")

        predictions = model(images).argmax(dim=1)
        matches = predictions == labels
        correct += matches.sum().item()
        total += labels.size(0)
        labels_cpu = labels.cpu()
        class_total += torch.bincount(labels_cpu, minlength=NUM_CLASSES)
        class_correct += torch.bincount(labels_cpu[matches.cpu()], minlength=NUM_CLASSES)

if total == 0:
    raise ValueError("Validation set is empty.")

val_accuracy = correct / total
print(f"Validation accuracy: {val_accuracy:.2%} ({correct}/{total})")

class_results = [
    (class_id, class_correct[class_id].item(), class_total[class_id].item())
    for class_id in range(NUM_CLASSES)
    if class_total[class_id].item() > 0
]
for heading, ranked_classes in (
    ("Top 10 classes", sorted(class_results, key=lambda result: (-result[1] / result[2], result[0]))[:10]),
    ("Bottom 10 classes", sorted(class_results, key=lambda result: (result[1] / result[2], result[0]))[:10]),
):
    print(f"\n{heading}:")
    for class_id, class_hits, class_count in ranked_classes:
        print(
            f"  {dataset.classes[class_id]}: "
            f"{class_hits / class_count:.2%} ({class_hits}/{class_count})"
        )

In [ ]:
def evaluate_per_class(eval_model):
    eval_model.eval()
    hits = torch.zeros(NUM_CLASSES, dtype=torch.int64)
    counts = torch.zeros(NUM_CLASSES, dtype=torch.int64)
    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=DEVICE == "cuda")
            matches = eval_model(images).argmax(dim=1).cpu() == labels
            counts += torch.bincount(labels, minlength=NUM_CLASSES)
            hits += torch.bincount(labels[matches], minlength=NUM_CLASSES)
    return hits, counts


def report_pruned_model(tag, pruned_model):
    """Prints accuracy, size, and per-class results against the baseline run above."""
    hits, counts = evaluate_per_class(pruned_model)
    num_correct, num_total = hits.sum().item(), counts.sum().item()
    print(f"\n===== {tag} =====")
    print(
        f"Validation accuracy: {num_correct / num_total:.2%} ({num_correct}/{num_total}) "
        f"| baseline {val_accuracy:.2%}"
    )

    num_params = sum(p.numel() for p in pruned_model.parameters())
    num_nonzero = sum(torch.count_nonzero(p).item() for p in pruned_model.parameters())
    size_bytes = sum(t.numel() * t.element_size() for t in pruned_model.state_dict().values())
    print(f"Parameters: {num_params:,} (baseline {total_params:,}, {num_params / total_params - 1:+.2%})")
    print(f"Nonzero parameters: {num_nonzero:,} ({num_nonzero / num_params:.2%} of parameters)")
    print(
        f"State dict tensor size: {size_bytes / 1024**2:.2f} MiB "
        f"(baseline {state_dict_size_mib:.2f} MiB, {size_bytes / state_dict_size_bytes - 1:+.2%})"
    )

    results = [
        (
            class_id,
            hits[class_id].item() / counts[class_id].item(),
            class_correct[class_id].item() / class_total[class_id].item(),
        )
        for class_id in range(NUM_CLASSES)
        if counts[class_id].item() > 0
    ]
    for heading, ranked_classes in (
        ("Top 10 classes", sorted(results, key=lambda result: (-result[1], result[0]))[:10]),
        ("Bottom 10 classes", sorted(results, key=lambda result: (result[1], result[0]))[:10]),
    ):
        print(f"\n{heading} (pruned | baseline | change):")
        for class_id, accuracy, baseline_accuracy in ranked_classes:
            print(
                f"  {dataset.classes[class_id]}: {accuracy:.2%} | {baseline_accuracy:.2%} "
                f"| {(accuracy - baseline_accuracy) * 100:+.2f}%p"
            )

## 4. Magnitude Based Pruning

In [ ]:
import torch.nn as nn

# timm ViT: blocks.{i}.mlp.fc1 (768 -> 3072) is the MLP hidden layer; add ".mlp.fc2" to prune the projection too.
MLP_LAYER_SUFFIXES = (".mlp.fc1",)


def get_mlp_hidden_linear_modules(model):
    """Pruning targets: Linear weights of the MLP hidden layer in each ViT block.
    Excludes attention, patch embedding, head, LayerNorm parameters, and biases."""
    return {
        n: m for n, m in model.named_modules()
        if isinstance(m, nn.Linear) and n.startswith("blocks.") and n.endswith(MLP_LAYER_SUFFIXES)
    }

def compute_prune_mask_per_output_channel(importance, ratio):
    """importance: (out_features, in_features); larger values mean more important (retain).
    In each output neuron (row), mark the lowest-scoring ratio fraction False.
    False indicates a weight to prune."""
    if not (0.0 <= ratio <= 1.0):
        raise ValueError(f"ratio must be in [0,1], got {ratio}")
    c_out = importance.shape[0]
    flat = importance.reshape(c_out, -1)
    row_len = flat.shape[1]
    n_prune = int(round(row_len * ratio))
    mask = torch.ones_like(flat, dtype=torch.bool)
    if n_prune > 0:
        _, prune_idx = torch.topk(flat, n_prune, largest=False, dim=1)
        mask.scatter_(1, prune_idx, False)
    return mask.view_as(importance)

def apply_prune_mask(module, mask):
    with torch.no_grad():
        module.weight.mul_(mask.to(dtype=module.weight.dtype, device=module.weight.device))

def prune_mlp_magnitude_(model, importance_fn, ratio):
    """Magnitude driver: accepts importance_fn(name, module) -> Tensor and
    applies the same pruning procedure to all MLP hidden Linear layers."""
    for name, module in get_mlp_hidden_linear_modules(model).items():
        importance = importance_fn(name, module)
        mask = compute_prune_mask_per_output_channel(importance, ratio)
        apply_prune_mask(module, mask)

In [ ]:
import copy

prune_ratio = [0.3, 0.5, 0.7]

for ratio in prune_ratio:
    pruned_model = copy.deepcopy(model)  # Keep the baseline model intact.
    prune_mlp_magnitude_(
        pruned_model,
        lambda name, module: module.weight.detach().abs(),
        ratio,
    )
    report_pruned_model(f"Magnitude pruning (ratio={ratio})", pruned_model)

## 5. Variance-Based Pruning

In [ ]:
from tqdm.auto import tqdm


class Welford:
    def __init__(self, device, hidden_dim: int):
        self.count = 0
        self.mean = torch.zeros(hidden_dim, dtype=torch.float64, device=device)
        self.M2 = torch.zeros(hidden_dim, dtype=torch.float64, device=device)

    def update(self, batch: torch.Tensor):
        batch = batch.to(dtype=torch.float64)
        batch_count = batch.size(0)
        batch_mean = torch.mean(batch, dim=0)

        total_count = self.count + batch_count

        delta = batch_mean - self.mean
        self.mean += delta * (batch_count / total_count)

        batch_var = batch.var(dim=0, unbiased=False)
        self.M2 += batch_var * batch_count + (delta ** 2) * self.count * batch_count / total_count

        self.count = total_count

    def finalize(self):
        if self.count < 2:
            return self.mean.float(), torch.zeros_like(self.mean).float()
        variance = self.M2 / (self.count - 1)
        return self.mean.float(), variance.float()


def score(model, dataloader, device, num_batches=0):
    stats = {}
    hooks = []

    def get_hook(layer_id):
        def hook(module, input, output):
            batch_size, num_tokens, hidden_dim = output.size()
            flattened = output.view(-1, hidden_dim)
            stats[layer_id].update(flattened)
        return hook

    for b, blk in enumerate(model.blocks):
        layer_id = f"0.{b}.hidden"
        hidden_dim = blk.mlp.fc1.out_features
        stats[layer_id] = Welford(device, hidden_dim)
        hooks.append(blk.mlp.act.register_forward_hook(get_hook(layer_id)))

    model.to(device)
    model.eval()

    with torch.no_grad():
        for batch_idx, (input_batch, _) in enumerate(
            tqdm(dataloader, desc="Collecting activation statistics",
                 total=num_batches if num_batches > 0 else None)
        ):
            if 0 < num_batches <= batch_idx:
                break
            input_batch = input_batch.to(device)
            model(input_batch)

    for h in hooks:
        h.remove()

    variances = {}
    means = {}
    for k, w in stats.items():
        mean, var = w.finalize()
        variances[k] = var
        means[k] = mean
    return variances, means


def threshold(scores_dict, percentage):
    all_scores = torch.cat([v.to("cpu") for v in scores_dict.values()])
    t = torch.quantile(all_scores, percentage)
    return t.item()


def indices(scores_dict, t):
    pruned = {}
    for k, v in scores_dict.items():
        mask = v <= t
        pruned[k] = torch.nonzero(mask, as_tuple=False).view(-1).tolist()
    return pruned


def prune(model, pruned, means):
    for layer_id, idx in pruned.items():
        if not idx:
            continue

        _, b_str, _ = layer_id.split(".")
        mlp = model.blocks[int(b_str)].mlp
        fc1 = mlp.fc1
        fc2 = mlp.fc2
        device = fc1.weight.device

        W1 = fc1.weight.data
        b1 = fc1.bias.data
        W2 = fc2.weight.data
        b2 = fc2.bias.data

        mean = means[layer_id][idx].to(device)
        b2 += W2[:, idx] @ mean

        mask_W1 = torch.ones(W1.size(0), dtype=torch.bool, device=device)
        mask_W2 = torch.ones(W2.size(1), dtype=torch.bool, device=device)
        mask_W1[idx] = False
        mask_W2[idx] = False

        W1 = W1[mask_W1]
        b1 = b1[mask_W1]
        W2 = W2[:, mask_W2]

        fc1.weight.data = W1
        fc1.bias.data = b1
        fc1.out_features = W1.size(0)

        fc2.weight.data = W2
        fc2.bias.data = b2
        fc2.in_features = W2.size(1)

    return model

In [ ]:
import copy

prune_ratio = [0.3, 0.5, 0.7]

variances, means = score(model, calibration_loader, DEVICE)   # 통계는 한 번만 수집

for ratio in prune_ratio:
    t = threshold(variances, percentage=ratio)
    pruned_idx = indices(variances, t)
    pruned_model = prune(copy.deepcopy(model), pruned_idx, means)  # 원본 model은 보존

    report_pruned_model(f"Variance-based pruning (ratio={ratio})", pruned_model)